In [63]:
import os  # for file operations

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    ArrayType,
)
from pyspark.sql.functions import (
    col,
    when,
    sum as spark_sum,
    round as spark_round,
    regexp_replace,
    split,
    udf,
    expr,
    explode,
)


# Setting our constants

In [64]:
ROOT_DIR = ""
PATH_PREFIX = "mnt/data/public/insideairbnb/data.insideairbnb.com/"

START_DATE = "2019-01-01"
END_DATE = "2025-12-31"

CITY_CONFIGS = [
    {
        "label": "new-york-city",
        "root_path": f"{PATH_PREFIX}united-states/ny/new-york-city",
        "regex_key": "united-states/ny/new-york-city",
    },
    {
        "label": "los-angeles",
        "root_path": f"{PATH_PREFIX}united-states/ca/los-angeles",
        "regex_key": "united-states/ca/los-angeles",
    },
    {
        "label": "chicago",
        "root_path": f"{PATH_PREFIX}united-states/il/chicago",
        "regex_key": "united-states/tx/austin",
    },
]

# Loading the Dataset

The Inside Airbnb dataset is organized by month, so each city has a separate folder for every period. To analyze the data, we first need to gather all the months post-pandemic, which we have to define to be after 2021. For each month, we read `listings.csv` to get the price information and `listings.csv.gz` to get the amenities offered by each listing. The function `build_city_airbnb_df` combines all these months into a single Spark DataFrame for each city and save the combined data as a Parquet file for later use.

In [65]:
def build_city_airbnb_df(
    city_config,
    start_year=2021,
    base_dir="/mnt/data/public/insideairbnb/data.insideairbnb.com/",
    save_parquet=True,
    parquet_dir="data/initial_join/",
):
    """
    Build a combined Spark DataFrame for a single city, joining the visualisation and data files for all months greater than the start_year.

    Args:
        city_config (dict): {'label': str, 'regex_key': str}
        start_year (int): starting year to consider
        base_dir (str): base directory of the Airbnb data
        save_parquet (bool): if True, saves the combined dataframe to parquet
        parquet_dir (str): directory to save parquet

    Returns:
        Spark DataFrame: combined city dataframe where for each id, it has the price and the amenities
    """

    spark = SparkSession.builder.appName(f"Airbnb_{city_config['label']}").getOrCreate()

    city_name = city_config["label"]
    city_base_dir = os.path.join(base_dir, city_config["regex_key"])
    output_path = os.path.join(parquet_dir, city_name)

    # If parquet exists, just load it
    if save_parquet and os.path.exists(output_path):
        return spark.read.parquet(output_path)

    # Get folders starting from start_year
    folders = [
        f
        for f in os.listdir(city_base_dir)
        if f[:4].isdigit() and int(f[:4]) >= start_year
    ]

    joined_frames = []

    # for each folder, get the price and amenities based on the unique id
    for folder in sorted(folders):
        month_path = os.path.join(city_base_dir, folder)
        data_filepath = os.path.join(month_path, "data/listings.csv.gz")
        viz_filepath = os.path.join(month_path, "visualisations/listings.csv")

        # read the viz
        df_viz = (
            spark.read.option("header", True)
            .csv(viz_filepath)
            .select("id", "price")
        )

        # read the data
        df_data = (
            spark.read.option("header", True)
            .option("inferSchema", True)
            .csv(data_filepath)
            .select("id", "amenities")
        )

        # join the columns
        df_joined = df_viz.join(df_data, on="id", how="left")

        joined_frames.append(df_joined)

    # Union all months
    final_df = joined_frames[0]
    for df in joined_frames[1:]:
        final_df = final_df.unionByName(df)

    # Save parquet if needed
    if save_parquet:
        os.makedirs(parquet_dir, exist_ok=True)
        final_df.write.mode("overwrite").parquet(output_path)
        print(f"Saved combined parquet to: {output_path}")

    return final_df

In [66]:
for city in CITY_CONFIGS:
    print(city['label'].replace('-', ' ').title())
    build_city_airbnb_df(city, save_parquet=True).show()

New York City
+-------------------+-----+--------------------+
|                 id|price|           amenities|
+-------------------+-----+--------------------+
| 816783428767938211|  110|"[""Air condition...|
|            8686040| NULL|"[""Air condition...|
|           12426778| NULL|                  []|
|           22671517| NULL|                NULL|
|           35776956| NULL|          "[""Pool""|
|    1min to subway"|   30|                NULL|
|           52331775| NULL|          "[""Oven""|
|1081507139395912406|   55|"[""Air condition...|
|            7389120|   99|          "[""Oven""|
| 786149074369437257|  152|"[""Air condition...|
| 777572930730588568|  500|"[""Security came...|
|1047936827694607886|   54|"[""Air condition...|
| 977732692492896948|  321|          "[""Oven""|
| 629211160504994365|   36|"[""Air condition...|
|            6939597|  120|                NULL|
| 681504125966887847|   50|Private room in home|
|           20481814|   57|"[""Paid street p...|
| 5889

# Checking for Null Values

For each city, we see that some listings have missing values in either the price or amenities columns. Since price is our target variable and amenities is a key feature for analysis, any row missing one of these values is not useful. Therefore, we remove rows that are empty in either column to ensure the dataset is complete and reliable for downstream analysis or modeling. After dropping null values in price or amenities, NYC retains 355,392 listings (67%), LA retains 380,924 listings (73%), and Chicago retains 103,685 listings (67%), which is more than sufficient for robust analysis and modeling.

In [67]:
for city in CITY_CONFIGS:
    print(city['label'].replace('-', ' ').title())
    city_df = build_city_airbnb_df(city, save_parquet=True)
    total = city_df.count()
    remaining = city_df.dropna().count()
    print(f"Initial Size of {total} (100%)")
    print(f"Non-empty rows of {remaining} ({int(round(remaining/total, 2) * 100)}%).")
    total_rows = city_df.count()  # total number of rows
    
    na_counts_percentage = city_df.select(
        [
            (spark_round(
                1-(spark_sum(when(col(c).isNull(), 1).otherwise(0)) / total_rows),
                2
            ) * 100).alias(f'{c} (%)')
            for c in city_df.columns
        ]
    )
    
    na_counts_percentage.show()

New York City
Initial Size of 527539 (100%)
Non-empty rows of 355392 (67%).
+------+---------+-------------+
|id (%)|price (%)|amenities (%)|
+------+---------+-------------+
| 100.0|     88.0|         77.0|
+------+---------+-------------+

Los Angeles
Initial Size of 524197 (100%)
Non-empty rows of 380924 (73%).
+------+---------+-------------+
|id (%)|price (%)|amenities (%)|
+------+---------+-------------+
| 100.0|     95.0|         77.0|
+------+---------+-------------+

Chicago
Initial Size of 154935 (100%)
Non-empty rows of 103685 (67%).
+------+---------+-------------+
|id (%)|price (%)|amenities (%)|
+------+---------+-------------+
| 100.0|     89.0|         76.0|
+------+---------+-------------+



# Cleaning the amenities

Given the cleaned city DataFrames, we now focus on processing the amenities column. The goal is to transform it into an array where each element represents a single amenity. This format is ideal for downstream tasks such as Word2Vec, which treats each row as a “sentence” and each amenity as a “word.”

During this cleaning process, some rows were removed because they contained invalid data. For example, occasionally a single number or unrelated text was present instead of actual amenities. These entries were dropped to ensure the dataset only contains valid amenities information.

After cleaning, the remaining number of listings for each city is as follows:
- New York City: 342,329 (64%) listings 
- Los Angeles: 367,000 (70%) listings
- Chicago: 98,989 (64%) listings

This ensures that each city’s dataset is clean, structured, and ready for Word2Vec or other analyses.

In [68]:
def robust_clean_amenities(arr):
    # Flatten array and clean each element
    cleaned = []
    for a in arr:
        if a:
            # Remove extra quotes, brackets, and escape chars
            a = a.replace('"', '').replace('[', '').replace(']', '').replace('\\u2013', '-').strip()
            if a:  # only keep non-empty
                cleaned.append(a)
    return cleaned

clean_udf = udf(robust_clean_amenities, ArrayType(StringType()))

def clean_city_amenities_robust(city_config, save_parquet=True, output_dir="data/cleaned_array"):
    city_df = build_city_airbnb_df(city_config).dropna()
    city_label = city_config['label']
    filepath = f'{output_dir}/{city_label}.parquet'
    if os.path.exists(filepath):
        return spark.read.parquet(filepath)
    
    # Remove outer brackets and extra quotes
    df_clean = city_df.withColumn(
        "amenities_clean", regexp_replace(col("amenities"), r'^-?\[+|\"+|\]+$', '')
    )
    
    # Split by commas
    df_clean = df_clean.withColumn("amenities_array", split(col("amenities_clean"), ","))
    
    # Apply robust cleaning UDF
    df_clean = df_clean.withColumn("amenities_array", clean_udf(col("amenities_array")))
    
    # Filter out invalid rows (single numeric value)
    df_clean = df_clean.filter(
        ~(
            (expr("size(amenities_array) = 1") & 
             (expr("amenities_array[0] RLIKE '^-?[0-9]+'") | 
              expr("amenities_array[0] = '-'")))
        )
    )

    
    # Save if requested
    if save_parquet:
        df_clean = df_clean.select(['id', 'price', 'amenities_array'])
        df_clean.write.mode("overwrite").parquet(filepath)
        print(f"Saved cleaned data for {city_label} to {filepath}")
    
    return df_clean

In [69]:
for city in CITY_CONFIGS:
    city_name = city['label'].replace('-', ' ').title()
    print(f"Cleaning amenities for {city_name}")
    df_cleaned = clean_city_amenities_robust(city, save_parquet=True)
    print(f"Cleaned rows of {df_cleaned.dropna().count()}")
    df_cleaned.show()


Cleaning amenities for New York City
Cleaned rows of 342329
+-----+-----+--------------------+
|   id|price|     amenities_array|
+-----+-----+--------------------+
| 5136|  175|          [Cable TV]|
| 5178|   61|        [Hair dryer]|
| 6872|   65|[Paid parking gar...|
| 7097|  199|        [Essentials]|
| 7750|   96|          [Cable TV]|
| 8490|  120|    [Cooking basics]|
| 9357|  150|['email', 'phone'...|
| 9657|  175|        [Essentials]|
|10452|   70|        [Essentials]|
|11943|  150|            [Washer]|
|12192|   40|        [Essentials]|
|12343|  250|          [Cable TV]|
|15385|   80|        [Essentials]|
|15396|  400| [Cleaning products]|
|15711|  206|        [High chair]|
|16595|  400| [Cleaning products]|
|16821|   99|        [Essentials]|
|16974|  250|        [Essentials]|
|17037|  160|        [Essentials]|
|18195|   70|        [Essentials]|
+-----+-----+--------------------+
only showing top 20 rows

Cleaning amenities for Los Angeles
Cleaned rows of 367000
+---------------

In [70]:
from pyspark.sql.functions import explode, col

# Read cleaned Parquet files
df_chicago = spark.read.parquet('data/cleaned_array/chicago.parquet')
df_ny = spark.read.parquet('data/cleaned_array/new-york-city.parquet')
df_la = spark.read.parquet('data/cleaned_array/los-angeles.parquet')

# Explode each DataFrame
df_chicago_exploded = df_chicago.select(explode(col("amenities_array")).alias("amenity"))
df_ny_exploded = df_ny.select(explode(col("amenities_array")).alias("amenity"))
df_la_exploded = df_la.select(explode(col("amenities_array")).alias("amenity"))

# Union all exploded DataFrames
df_all = df_chicago_exploded.union(df_ny_exploded).union(df_la_exploded)

# Get unique amenities across all cities
unique_amenities = df_all.distinct()

print(f"Number of unique amenities across Chicago, NYC, and LA: {unique_amenities.count()}")


Number of unique amenities across Chicago, NYC, and LA: 2399


In [71]:
df_chicago_exploded.distinct().count()

719

In [72]:
df_ny_exploded.distinct().count()

1114

In [73]:
df_la_exploded.distinct().count()

1310